In [33]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as imbpipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import (
    train_test_split ,
    StratifiedKFold ,
    cross_val_score ,
    GridSearchCV
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [34]:
df = pd.read_csv("car_data_augmented.csv")
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [35]:
df.drop(columns = "Car_Name" , inplace = True)
X = df.drop(columns = "Owner")
y = df["Owner"]
print("\nClass Distribution")
print(y.value_counts(normalize = True))


Class Distribution
Owner
0    0.964523
1    0.031042
3    0.004435
Name: proportion, dtype: float64


In [36]:
count = y.value_counts()
correct_count = count[count >= 5].index
valid = y.isin(correct_count)
X_clean = X[valid].copy()
y_clean = y[valid].copy()
X_train , X_test , y_train , y_test = train_test_split(X_clean , y_clean , test_size = 0.2, random_state = 42 , stratify = y_clean)
categirical_columns = list(X.select_dtypes(include = ["object"]))
numerical_columns = list(X.select_dtypes(exclude = ["object"]))

In [37]:
numerical_transform = Pipeline([
    ("scaler" , StandardScaler())
])
categorical_transform = Pipeline([
    ("encoder" , OneHotEncoder(handle_unknown ="ignore"))
])
preprocessor = ColumnTransformer([
    ("num" , numerical_transform ,numerical_columns ),
    ("cat" , categorical_transform , categirical_columns)
])
log_pipeline = imbpipeline([
    ("preprocessor" , preprocessor),
    ("smote" , SMOTE(random_state = 42)),
    ("classifier" , LogisticRegression(max_iter = 500))
])
rf_pipeline = imbpipeline([
    ("preprocessor" , preprocessor),
    ("smote" , SMOTE(random_state = 42)),
    ("classifier" , RandomForestClassifier(random_state = 42))
])
cv = StratifiedKFold(
    n_splits = 5 ,
    shuffle = True ,
    random_state = 42
)
log_cv = cross_val_score(
    log_pipeline ,
    X_train,
    y_train,
    cv = cv,
    scoring = "roc_auc"
)
rf_cv = cross_val_score(
    rf_pipeline ,
    X_train ,
    y_train ,
    cv = cv,
    scoring = "roc_auc"
    )
param_grid = {
    "classifier__n_estimators":[100 , 200],
    "classifier__max_depth":[5,10,None],
    "classifier__min_samples_split":[2,5]
}
grid = GridSearchCV(
    rf_pipeline ,
    param_grid ,
    scoring = "roc_auc",
    n_jobs = -1
)
grid.fit(X_train , y_train)
print("\nThe best prmtrs are")
print(grid.best_params_)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)


The best prmtrs are
{'classifier__max_depth': None, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}


In [39]:
print("\nAccuracy:",
      accuracy_score(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))


Accuracy: 0.9555555555555556

Classification Report
              precision    recall  f1-score   support

           0       0.97      0.99      0.98        87
           1       0.00      0.00      0.00         3

    accuracy                           0.96        90
   macro avg       0.48      0.49      0.49        90
weighted avg       0.93      0.96      0.94        90

